# Day 29/30 — Building an End-to-End Data Analytics Project
## Synthetic Citizen-Grievance Analytics | 30-Day Data Analytics Cookbook

**Objective:** Build a reproducible workflow from business questions → synthetic data → validation → cleaning → KPIs → EDA → forecasting baseline → evaluation → executive recommendations.

**Important:** All records are synthetic, with no real citizens or operational claims. Outputs are written to a local `day29_outputs/` directory. Execute cells in order.

## 1. Business questions and KPI definitions
1. How many grievances were received, resolved, and pending?
2. Which zones, departments, and categories have the greatest unresolved workload?
3. How does monthly intake compare with monthly closures?
4. What is the typical time to resolution?
5. Does a simple one-month-lag baseline predict monthly complaint intake adequately?

**Definitions:** *Received* is based on registration date. *Resolved* means status `RESOLVED` and a valid closure date. *Pending* includes all other valid records. Monthly closures are counted by **closure date**, so monthly received-minus-closed is a net flow, not necessarily the current backlog. Resolution time is measured in elapsed days for resolved records. The synthetic dataset has no reopened cases.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 29
rng = np.random.default_rng(SEED)
OUT = Path('day29_outputs')
OUT.mkdir(exist_ok=True)
plt.rcParams['figure.figsize'] = (10, 5)
print('Outputs:', OUT.resolve())

## 2. Generate a realistic synthetic dataset
We simulate 24 months of grievances with modest upward trend, monsoon-season effects, different zone/category mixes, and plausible closure delays. We intentionally inject duplicate IDs, inconsistent status formatting, and invalid dates to demonstrate a quality-control workflow.

In [ ]:
months = pd.date_range('2024-01-01', periods=24, freq='MS')
zones = ['North', 'Central', 'South', 'West']
departments = ['Public Health', 'Solid Waste', 'Roads', 'Stormwater']
complaint_types = ['Sanitation', 'Garbage', 'Pothole', 'Waterlogging']
records = []
seq = 0
for i, month in enumerate(months):
    expected = 105 + i * 2.2 + (35 if month.month in [10, 11, 12] else 0)
    count = int(rng.poisson(expected))
    for _ in range(count):
        seq += 1
        day = int(rng.integers(1, month.days_in_month + 1))
        registered = month + pd.Timedelta(days=day - 1)
        zone = str(rng.choice(zones, p=[0.29, 0.24, 0.30, 0.17]))
        category_idx = int(rng.choice(4, p=[0.28, 0.32, 0.23, 0.17]))
        dept = departments[category_idx]
        category = complaint_types[category_idx]
        # Older records have had more time to close; use a fixed observation cutoff.
        days_to_close = max(0, int(rng.gamma(shape=2.0, scale=4.5)))
        potential_close = registered + pd.Timedelta(days=days_to_close)
        cutoff = pd.Timestamp('2025-12-31')
        resolved = (rng.random() < (0.89 if category != 'Waterlogging' else 0.81)) and potential_close <= cutoff
        records.append({
            'complaint_id': f'SYN-{seq:06d}',
            'complaint_date': registered.strftime('%Y-%m-%d'),
            'zone': zone,
            'department': dept,
            'complaint_type': category,
            'status': 'RESOLVED' if resolved else 'PENDING',
            'closed_date': potential_close.strftime('%Y-%m-%d') if resolved else None
        })
raw = pd.DataFrame(records)
# Inject demonstrable quality issues into a COPY, retaining the original clean source.
noisy = raw.copy()
noisy.loc[0, 'status'] = ' resolved '
noisy.loc[1, 'complaint_date'] = 'not-a-date'
noisy.loc[2, 'closed_date'] = '1900-01-01'
noisy = pd.concat([noisy, noisy.iloc[[3]]], ignore_index=True)
noisy.to_csv(OUT / 'synthetic_grievances_raw.csv', index=False)
print('Synthetic source rows:', len(raw), '| With injected issues:', len(noisy))
noisy.head()

## 3. Data quality audit and cleaning
The cleaning stage preserves an audit trail. Duplicate IDs are removed; invalid registration dates are excluded; inconsistent closure/status combinations are converted to pending and flagged. Real projects should apply agency-approved business rules rather than silently overwriting questionable records.

In [ ]:
df = noisy.copy()
df.columns = df.columns.str.strip().str.lower()
quality = {}
quality['input_rows'] = len(df)
quality['duplicate_ids'] = int(df.duplicated('complaint_id').sum())
df = df.drop_duplicates('complaint_id', keep='first').copy()
df['status'] = df['status'].astype('string').str.strip().str.upper()
df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')
quality['invalid_registration_dates'] = int(df['complaint_date'].isna().sum())
df = df.dropna(subset=['complaint_date']).copy()
invalid_close = (df['status'].eq('RESOLVED').fillna(False) &
                 (df['closed_date'].isna() | (df['closed_date'] < df['complaint_date'])))
quality['invalid_resolved_closure_dates'] = int(invalid_close.sum())
df.loc[invalid_close, 'status'] = 'PENDING'
df.loc[invalid_close, 'closed_date'] = pd.NaT
# No closure dates are retained for pending complaints.
df.loc[df['status'] != 'RESOLVED', 'closed_date'] = pd.NaT
df['is_resolved'] = df['status'].eq('RESOLVED').fillna(False)
df['resolution_days'] = (df['closed_date'] - df['complaint_date']).dt.total_seconds() / 86400
df['registration_month'] = df['complaint_date'].dt.to_period('M').dt.to_timestamp()
df['closure_month'] = df['closed_date'].dt.to_period('M').dt.to_timestamp()
quality['clean_rows'] = len(df)
quality_report = pd.Series(quality, name='count')
quality_report.to_csv(OUT / 'quality_audit.csv')
df.to_csv(OUT / 'synthetic_grievances_clean.csv', index=False)
assert df['complaint_id'].is_unique
assert df['complaint_date'].notna().all()
assert (df.loc[df['is_resolved'], 'resolution_days'] >= 0).all()
quality_report

## 4. Executive KPIs
**Snapshot cutoff:** 31 December 2025. Resolution rate is the share of registered grievances marked resolved by that date. This differs from a monthly closure-to-intake ratio.

In [ ]:
total = len(df)
resolved = int(df['is_resolved'].sum())
pending = total - resolved
resolution_rate = 100 * resolved / total
avg_resolution_days = df.loc[df['is_resolved'], 'resolution_days'].mean()
median_resolution_days = df.loc[df['is_resolved'], 'resolution_days'].median()
kpis = pd.DataFrame({'Metric': ['Total registered', 'Resolved', 'Pending', 'Resolution rate (%)', 'Mean resolution days', 'Median resolution days'],
                     'Value': [total, resolved, pending, round(resolution_rate, 2), round(avg_resolution_days, 2), round(median_resolution_days, 2)]})
kpis.to_csv(OUT / 'kpi_summary.csv', index=False)
kpis

## 5. Trend analysis: monthly intake versus closures
Closure counts are grouped by actual closure month; no claim is made that a month’s closures belong to the same month’s registrations.

In [ ]:
received_by_month = df.groupby('registration_month').size().rename('received')
closed_by_month = df.loc[df['is_resolved']].groupby('closure_month').size().rename('closed')
monthly = pd.concat([received_by_month, closed_by_month], axis=1).fillna(0).astype(int)
monthly = monthly.reindex(months, fill_value=0)
monthly.index.name = 'month'
monthly['net_flow'] = monthly['received'] - monthly['closed']
monthly.to_csv(OUT / 'monthly_summary.csv')
ax = monthly[['received', 'closed']].plot(marker='o', markersize=3, title='Synthetic grievances: received vs closed')
ax.set_ylabel('Complaints')
ax.set_xlabel('Month')
plt.tight_layout()
plt.savefig(OUT / 'monthly_received_closed.png', dpi=160)
plt.show()
monthly.tail(6)

## 6. Zone and complaint-type performance
Prioritize by absolute pending workload as well as resolution rate; small groups may have unstable rates.

In [ ]:
zone = df.groupby('zone').agg(total=('complaint_id', 'size'), resolved=('is_resolved', 'sum'))
zone['pending'] = zone['total'] - zone['resolved']
zone['resolution_rate_pct'] = (100 * zone['resolved'] / zone['total']).round(1)
zone = zone.sort_values('pending', ascending=False)
category = df.groupby('complaint_type').agg(total=('complaint_id', 'size'), resolved=('is_resolved', 'sum'))
category['pending'] = category['total'] - category['resolved']
category['resolution_rate_pct'] = (100 * category['resolved'] / category['total']).round(1)
category = category.sort_values('pending', ascending=False)
zone.to_csv(OUT / 'zone_summary.csv')
category.to_csv(OUT / 'category_summary.csv')
fig, ax = plt.subplots()
zone['pending'].sort_values().plot(kind='barh', ax=ax)
ax.set_title('Pending grievances by zone (synthetic)')
ax.set_xlabel('Pending complaints')
plt.tight_layout()
plt.savefig(OUT / 'pending_by_zone.png', dpi=160)
plt.show()
print('Zone performance:')
print(zone.to_string())
print('\nCategory performance:')
print(category.to_string())

## 7. Cohort-based resolution performance
For an apples-to-apples comparison, examine resolution outcomes by **registration month**. The most recent cohorts are right-censored: they have had less time to close, so interpret their resolution rates cautiously.

In [ ]:
cohorts = df.groupby('registration_month').agg(received=('complaint_id', 'size'), resolved_to_cutoff=('is_resolved', 'sum'))
cohorts['resolution_rate_pct'] = (100 * cohorts['resolved_to_cutoff'] / cohorts['received']).round(1)
cohorts.to_csv(OUT / 'registration_cohorts.csv')
cohorts.tail(6)

## 8. Forecasting baseline and chronological backtest
A one-month-lag naive baseline predicts each month from the preceding month. Evaluate on the **last six months** only; the earlier period is historical context. For an operational model, compare this baseline against seasonal naive and other candidates on identical rolling-origin splits.

In [ ]:
forecast = monthly[['received']].copy()
forecast['naive_prediction'] = forecast['received'].shift(1)
test = forecast.iloc[-6:].dropna()
mae = mean_absolute_error(test['received'], test['naive_prediction'])
rmse = np.sqrt(mean_squared_error(test['received'], test['naive_prediction']))
bias = (test['naive_prediction'] - test['received']).mean()
metrics = pd.DataFrame({'metric': ['MAE', 'RMSE', 'Mean forecast bias'], 'value': [mae, rmse, bias]})
metrics.to_csv(OUT / 'forecast_metrics.csv', index=False)
ax = test.plot(y=['received', 'naive_prediction'], marker='o', title='Holdout: actual vs naive forecast')
ax.set_ylabel('Monthly complaints')
plt.tight_layout()
plt.savefig(OUT / 'forecast_holdout.png', dpi=160)
plt.show()
print(metrics.round(2).to_string(index=False))

## 9. Translate results into a management briefing
Recommendations are **hypotheses to validate**, not evidence of a specific operational cause. Suggested success indicators: backlog count, resolution time, and repeat-complaint volume. A decision owner should confirm resource availability and target feasibility.

In [ ]:
top_zone = zone.index[0]
top_category = category.index[0]
brief = f"""# Executive Brief — Synthetic Citizen Grievance Analytics

**Scope:** {months[0]:%b %Y} to {months[-1]:%b %Y}; synthetic records only.

## KPI snapshot
- Registered: {total:,}; resolved: {resolved:,}; pending: {pending:,}.
- Resolution rate: {resolution_rate:.1f}%.
- Median resolution time (resolved cases): {median_resolution_days:.1f} days.

## Priority signals
- Largest pending workload by zone: **{top_zone}** ({int(zone.loc[top_zone, 'pending']):,}).
- Largest pending workload by category: **{top_category}** ({int(category.loc[top_category, 'pending']):,}).
- Naive forecasting baseline (last six months): MAE **{mae:.1f}** complaints.

## Recommended next steps
1. Validate the top backlog segments with operational teams and check case age.
2. Investigate root causes and pilot targeted interventions in priority segments.
3. Review backlog, resolution time, and intake trends weekly; compare against an agreed baseline.

**Limitations:** Synthetic data, simplified statuses, no reopened cases, and recent-cohort censoring. No causal claims are established.
"""
(OUT / 'executive_brief.md').write_text(brief, encoding='utf-8')
print(brief)

## 10. Reproducibility and extension exercises
- Replace synthetic generation with a **de-identified** approved CSV extract, preserving the same schema.
- Add a data dictionary and document operational definitions of `RESOLVED`, `CLOSED`, and reopened cases.
- Add ward-level analysis and age-bucketed backlog (0–7, 8–30, >30 days).
- Compare seasonal-naive forecasts against the one-month baseline with rolling-origin backtests.
- Add automated assertions for missing dates, duplicates, impossible durations, and unexpected status values.
- Create a one-page decision brief with three findings, two interventions, and measurable outcomes.

**Privacy:** Never publish real citizen names, phone numbers, addresses, complaint narratives, or precise locations in a public repository.

**Core lesson:** A good analytics project links a business decision to trusted data, reproducible analysis, validation, and action.